In [2]:
# ============================================================
# WASTE GLASS POWDER CONCRETE
# COMPLETE MODELLING AND MANUSCRIPT REVISION
#
# FIVE ORIGINAL MODELS ONLY:
# 1. AdaBoost
# 2. Random Forest
# 3. Extra Trees
# 4. SVM / SVR
# 5. NGBoost
#
# Reviewer requirements:
# B1-B4, B6-B7
# C1
# D1-D7
#
# Python script: WGP_Five_Models_Optimized.py
# ============================================================

import sys
import os
import time
import json
import platform
import warnings
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import joblib

from scipy.stats import skew

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    RandomizedSearchCV,
    train_test_split,
    ParameterGrid
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import LogScore

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_PATH = Path(
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work"
    r"\Data\01_WGP_database_with_ids.csv"
)

OUTPUT_DIR = Path(
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work\Models"
)

SEED = 42

# Parallelize independent cross-validation fits; limit workers to avoid RAM pressure.
# Adjust to 2 for an 8 GB computer; 4 for 16 GB or more.
CV_WORKERS = min(4, max(1, (os.cpu_count() or 2) - 1))
# Prevent BLAS/NumPy from starting multiple threads in every worker.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# Final manuscript settings
N_ITER = 30
N_BOOTSTRAP = 2000

# For preliminary testing, use:
# N_ITER = 5
# N_BOOTSTRAP = 100

TARGET = "CS"
SOURCE = "source_id"
RECORD_ID = "record_id"

FEATURES = [
    "GGP size",
    "Replacement",
    "Cement",
    "Max size",
    "CA",
    "FA",
    "WC Ratio",
    "SiO2",
    "CaO",
    "Na2O",
    "Days"
]

np.random.seed(SEED)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 2. OUTPUT FOLDERS
# ============================================================

FOLDERS = {
    "models": OUTPUT_DIR / "Saved_Models",
    "predictions": OUTPUT_DIR / "Predictions",
    "tables": OUTPUT_DIR / "Tables",
    "figures": OUTPUT_DIR / "Figures",
    "supplement": OUTPUT_DIR / "Supplementary",
    "logs": OUTPUT_DIR / "Logs"
}

for folder in FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading WGP dataset...")

df = pd.read_csv(DATA_PATH)

required = FEATURES + [
    TARGET,
    SOURCE,
    RECORD_ID
]

missing = [
    col for col in required
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

if df[RECORD_ID].isna().any():
    raise ValueError("Missing record IDs detected.")

if df[RECORD_ID].duplicated().any():
    raise ValueError("Duplicate record IDs detected.")

if df[SOURCE].isna().any():
    raise ValueError("Missing source IDs detected.")

for col in FEATURES + [TARGET]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

if df[TARGET].isna().any():
    raise ValueError("Target contains missing values.")

if df[FEATURES].isna().all().any():
    raise ValueError(
        "At least one input feature is entirely missing."
    )

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[SOURCE].copy()

N = len(df)
N_SOURCES = groups.nunique()

print("Samples:", N)
print("Features:", len(FEATURES))
print("Sources:", N_SOURCES)

if N != 337:
    raise ValueError(
        f"Expected 337 records; found {N}."
    )

if N_SOURCES != 18:
    raise ValueError(
        f"Expected 18 sources; found {N_SOURCES}."
    )


# ============================================================
# 4. DEFINE FIVE MODELS
# ============================================================

def make_pipeline(model, scale=False):

    steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]

    if scale:
        steps.append(
            ("scaler", StandardScaler())
        )

    steps.append(("model", model))

    return Pipeline(steps)


MODELS = {

    "AdaBoost": {
        "pipeline": make_pipeline(
            AdaBoostRegressor(
                random_state=SEED
            )
        ),
        "params": {
            "model__n_estimators": [
                50, 100, 200, 400
            ],
            "model__learning_rate": [
                0.01, 0.05, 0.1, 0.5, 1.0
            ],
            "model__loss": [
                "linear",
                "square",
                "exponential"
            ]
        }
    },

    "Random_Forest": {
        "pipeline": make_pipeline(
            RandomForestRegressor(
                random_state=SEED,
                n_jobs=1
            )
        ),
        "params": {
            "model__n_estimators": [
                100, 200, 400
            ],
            "model__max_depth": [
                None, 5, 10, 20
            ],
            "model__min_samples_split": [
                2, 5, 10
            ],
            "model__min_samples_leaf": [
                1, 2, 4
            ],
            "model__max_features": [
                0.7, 1.0
            ]
        }
    },

    "Extra_Trees": {
        "pipeline": make_pipeline(
            ExtraTreesRegressor(
                random_state=SEED,
                n_jobs=1
            )
        ),
        "params": {
            "model__n_estimators": [
                100, 200, 400
            ],
            "model__max_depth": [
                None, 5, 10, 20
            ],
            "model__min_samples_split": [
                2, 5, 10
            ],
            "model__min_samples_leaf": [
                1, 2, 4
            ],
            "model__max_features": [
                0.7, 1.0
            ]
        }
    },

    "SVM": {
        "pipeline": make_pipeline(
            SVR(kernel="rbf"),
            scale=True
        ),
        "params": {
            "model__C": [
                0.1, 1, 10, 100, 1000
            ],
            "model__epsilon": [
                0.01, 0.1, 0.5, 1.0
            ],
            "model__gamma": [
                "scale",
                0.001,
                0.01,
                0.1,
                1.0
            ]
        }
    },

    "NGBoost": {
        "pipeline": make_pipeline(
            NGBRegressor(
                Dist=Normal,
                Score=LogScore,
                Base=DecisionTreeRegressor(
                    max_depth=3,
                    random_state=SEED
                ),
                random_state=SEED,
                verbose=False
            )
        ),
        "params": {
            "model__n_estimators": [
                100, 200, 350, 500
            ],
            "model__learning_rate": [
                0.005, 0.01, 0.03, 0.05
            ],
            "model__minibatch_frac": [
                0.5, 0.8, 1.0
            ],
            "model__col_sample": [
                0.6, 0.8, 1.0
            ],
            "model__natural_gradient": [
                True, False
            ]
        }
    }
}


# ============================================================
# 5. METRICS
# ============================================================

def calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    error = predicted - actual

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    mae = mean_absolute_error(
        actual, predicted
    )

    if len(actual) > 1 and np.var(actual) > 0:
        r2 = r2_score(actual, predicted)
    else:
        r2 = np.nan

    valid = np.abs(actual) > 1e-12

    if valid.any():

        relative_error = (
            np.abs(error[valid])
            / np.abs(actual[valid])
        )

        mape = 100 * np.mean(relative_error)

        within10 = 100 * np.mean(
            relative_error <= 0.10
        )

        within20 = 100 * np.mean(
            relative_error <= 0.20
        )

        within30 = 100 * np.mean(
            relative_error <= 0.30
        )

    else:

        mape = np.nan
        within10 = np.nan
        within20 = np.nan
        within30 = np.nan

    return {
        "N": len(actual),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_percent": mape,
        "Within_10_percent": within10,
        "Within_20_percent": within20,
        "Within_30_percent": within30,
        "Max_Overprediction": max(
            0.0, float(np.max(error))
        ),
        "Max_Underprediction": min(
            0.0, float(np.min(error))
        ),
        "Residual_Mean": float(np.mean(error)),
        "Residual_SD": (
            float(np.std(error, ddof=1))
            if len(error) > 1
            else np.nan
        ),
        "Residual_Skewness": (
            float(skew(error, bias=False))
            if len(error) > 2
            and np.std(error) > 0
            else np.nan
        )
    }


# ============================================================
# 6. HYPERPARAMETER TUNING
# ============================================================

def tune_model(
    configuration,
    X_train,
    y_train,
    group_train
):

    pipeline = clone(
        configuration["pipeline"]
    )

    params = configuration["params"]

    inner_cv = GroupKFold(
        n_splits=5
    )

    total_combinations = len(
        ParameterGrid(params)
    )

    candidates = min(
        N_ITER,
        total_combinations
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=candidates,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=SEED,
        n_jobs=CV_WORKERS,
        pre_dispatch=CV_WORKERS,
        refit=True,
        error_score="raise",
        return_train_score=False
    )

    search.fit(
        X_train,
        y_train,
        groups=group_train
    )

    return search


# ============================================================
# 7. PREDICTION RECORD FUNCTION
# ============================================================

def prediction_records(
    model_name,
    indices,
    predictions,
    split_name,
    fold=None,
    mu=None,
    sigma=None
):

    records = []

    for j, idx in enumerate(indices):

        actual = float(y.iloc[idx])
        predicted = float(predictions[j])

        records.append({
            "record_id": df[RECORD_ID].iloc[idx],
            "source_id": groups.iloc[idx],
            "model": model_name,
            "split": split_name,
            "outer_fold": fold,
            "y_true": actual,
            "y_pred": predicted,
            "error_pred_minus_actual": (
                predicted - actual
            ),
            "absolute_error": abs(
                predicted - actual
            ),
            "mu": (
                float(mu[j])
                if mu is not None
                else np.nan
            ),
            "sigma": (
                float(sigma[j])
                if sigma is not None
                else np.nan
            )
        })

    return records


# ============================================================
# 8. MAIN MODELLING
# ============================================================

outer_records = []
train_records = []
test_records = []
full_fit_records = []

tuning_records = []
search_space_records = []
final_parameter_records = []

outer_cv = LeaveOneGroupOut()

for model_name, configuration in MODELS.items():

    model_start = time.perf_counter()
    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    # --------------------------------------------------------
    # TABLE 3: SEARCH SPACE
    # --------------------------------------------------------

    total_combinations = len(
        ParameterGrid(configuration["params"])
    )

    for param, values in configuration["params"].items():

        search_space_records.append({
            "Model": model_name,
            "Hyperparameter": param,
            "Search_Values": str(values),
            "Total_Grid_Combinations": total_combinations,
            "Candidates_Per_Search": min(
                N_ITER,
                total_combinations
            )
        })

    # --------------------------------------------------------
    # NESTED LOSO
    # --------------------------------------------------------

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups),
        start=1
    ):

        heldout_source = groups.iloc[test_idx[0]]

        print(
            f"{model_name}: LOSO fold "
            f"{fold}/{N_SOURCES}, "
            f"source={heldout_source}",
            flush=True
        )

        search = tune_model(
            configuration,
            X.iloc[train_idx],
            y.iloc[train_idx],
            groups.iloc[train_idx]
        )

        fitted = search.best_estimator_

        predicted = fitted.predict(
            X.iloc[test_idx]
        )

        mu = None
        sigma = None

        if model_name == "NGBoost":

            transformed = fitted[:-1].transform(
                X.iloc[test_idx]
            )

            distribution = (
                fitted.named_steps["model"]
                .pred_dist(transformed)
            )

            mu = np.asarray(
                distribution.loc
            )

            sigma = np.asarray(
                distribution.scale
            )

        outer_records.extend(
            prediction_records(
                model_name,
                test_idx,
                predicted,
                "Nested_LOSO",
                fold=fold,
                mu=mu,
                sigma=sigma
            )
        )

        tuning_records.append({
            "Model": model_name,
            "Validation": "Nested_LOSO",
            "Outer_Fold": fold,
            "Heldout_Source": heldout_source,
            "Training_N": len(train_idx),
            "Testing_N": len(test_idx),
            "Best_Inner_RMSE": -search.best_score_,
            "Candidates_Evaluated": len(
                search.cv_results_["params"]
            ),
            "Best_Parameters": json.dumps(
                search.best_params_,
                default=str
            )
        })

    # --------------------------------------------------------
    # RANDOM 80:20 SPLIT
    # --------------------------------------------------------

    print(
        f"{model_name}: Random 80:20 comparison...",
        flush=True
    )

    indices = np.arange(N)

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        shuffle=True
    )

    search_80 = tune_model(
        configuration,
        X.iloc[train_idx],
        y.iloc[train_idx],
        groups.iloc[train_idx]
    )

    fitted_80 = search_80.best_estimator_

    predicted_train = fitted_80.predict(
        X.iloc[train_idx]
    )

    predicted_test = fitted_80.predict(
        X.iloc[test_idx]
    )

    train_records.extend(
        prediction_records(
            model_name,
            train_idx,
            predicted_train,
            "Random_80_Training"
        )
    )

    test_records.extend(
        prediction_records(
            model_name,
            test_idx,
            predicted_test,
            "Random_20_Testing"
        )
    )

    tuning_records.append({
        "Model": model_name,
        "Validation": "Random_80_20",
        "Outer_Fold": np.nan,
        "Heldout_Source": np.nan,
        "Training_N": len(train_idx),
        "Testing_N": len(test_idx),
        "Best_Inner_RMSE": -search_80.best_score_,
        "Candidates_Evaluated": len(
            search_80.cv_results_["params"]
        ),
        "Best_Parameters": json.dumps(
            search_80.best_params_,
            default=str
        )
    })

    # --------------------------------------------------------
    # FINAL FULL-DATABASE MODEL
    # --------------------------------------------------------

    print(
        f"{model_name}: Final full-database tuning...",
        flush=True
    )

    final_search = tune_model(
        configuration,
        X,
        y,
        groups
    )

    final_model = final_search.best_estimator_

    joblib.dump(
        final_model,
        FOLDERS["models"]
        / f"{model_name}_Final_Pipeline.joblib"
    )

    final_parameter_records.append({
        "Model": model_name,
        "Best_Inner_RMSE": -final_search.best_score_,
        "Candidates_Evaluated": len(
            final_search.cv_results_["params"]
        ),
        "Final_Selected_Parameters": json.dumps(
            final_search.best_params_,
            default=str
        )
    })

    full_predictions = final_model.predict(X)

    full_fit_records.extend(
        prediction_records(
            model_name,
            np.arange(N),
            full_predictions,
            "Full_Database_Training_Fit"
        )
    )

    # Persist this model's predictions and tuning logs immediately.
    pd.DataFrame([r for r in outer_records if r["model"] == model_name]).to_csv(
        FOLDERS["predictions"] / f"{model_name}_LOSO_checkpoint.csv", index=False
    )
    pd.DataFrame([r for r in tuning_records if r["Model"] == model_name]).to_csv(
        FOLDERS["logs"] / f"{model_name}_tuning_checkpoint.csv", index=False
    )
    print(f"{model_name}: COMPLETED in {(time.perf_counter()-model_start)/60:.1f} minutes", flush=True)


# ============================================================
# 9. SAVE ALL PREDICTIONS
# ============================================================

outer_df = pd.DataFrame(outer_records)
train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)
full_df = pd.DataFrame(full_fit_records)

outer_df.to_csv(
    FOLDERS["predictions"]
    / "outer_fold_predictions.csv",
    index=False
)

train_df.to_csv(
    FOLDERS["predictions"]
    / "random80_train_actual_predicted_error.csv",
    index=False
)

test_df.to_csv(
    FOLDERS["predictions"]
    / "random20_test_actual_predicted_error.csv",
    index=False
)

full_df.to_csv(
    FOLDERS["predictions"]
    / "full_database_training_fit_predictions.csv",
    index=False
)

search_space_df = pd.DataFrame(
    search_space_records
)

final_params_df = pd.DataFrame(
    final_parameter_records
)

tuning_df = pd.DataFrame(
    tuning_records
)

search_space_df.to_csv(
    FOLDERS["tables"]
    / "Table3_Search_Spaces.csv",
    index=False
)

final_params_df.to_csv(
    FOLDERS["tables"]
    / "Table3_Final_Selected_Parameters.csv",
    index=False
)

tuning_df.to_csv(
    FOLDERS["logs"]
    / "Hyperparameter_Tuning_Log.csv",
    index=False
)


# ============================================================
# 10. TABLE 5: PERFORMANCE
# ============================================================

table5_records = []

for model_name in MODELS:

    for validation, data in [
        ("Nested_LOSO", outer_df),
        ("Random_80_Training", train_df),
        ("Random_20_Testing", test_df)
    ]:

        subset = data[
            data["model"] == model_name
        ]

        metrics = calculate_metrics(
            subset["y_true"],
            subset["y_pred"]
        )

        table5_records.append({
            "Validation": validation,
            "Model": model_name,
            **metrics
        })

table5_df = pd.DataFrame(table5_records)


# ============================================================
# 11. SOURCE-CLUSTER BOOTSTRAP
# ============================================================

print("\nRunning cluster bootstrap...")

rng = np.random.default_rng(SEED)

unique_sources = groups.unique()

source_indices = {
    source: np.flatnonzero(
        groups.to_numpy() == source
    )
    for source in unique_sources
}

bootstrap_indices = []

for b in range(N_BOOTSTRAP):

    sampled_sources = rng.choice(
        unique_sources,
        size=N_SOURCES,
        replace=True
    )

    sampled_indices = np.concatenate([
        source_indices[source]
        for source in sampled_sources
    ])

    bootstrap_indices.append(sampled_indices)


# ============================================================
# 12. ALIGN LOSO PREDICTIONS
# ============================================================

ordered_predictions = {}

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    subset = (
        subset.set_index("record_id")
        .loc[df[RECORD_ID]]
    )

    ordered_predictions[model_name] = subset


# ============================================================
# 13. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

ci_records = []

for model_name in MODELS:

    subset = ordered_predictions[model_name]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    bootstrap_metrics = {
        "R2": [],
        "RMSE": [],
        "MAE": []
    }

    for idx in bootstrap_indices:

        metrics = calculate_metrics(
            actual[idx],
            predicted[idx]
        )

        for metric in bootstrap_metrics:
            bootstrap_metrics[metric].append(
                metrics[metric]
            )

    result = {"Model": model_name}

    for metric, values in bootstrap_metrics.items():

        result[f"{metric}_CI_Lower"] = (
            np.nanpercentile(values, 2.5)
        )

        result[f"{metric}_CI_Upper"] = (
            np.nanpercentile(values, 97.5)
        )

    ci_records.append(result)

ci_df = pd.DataFrame(ci_records)

table5_df = table5_df.merge(
    ci_df,
    on="Model",
    how="left"
)

ci_columns = [
    col for col in table5_df.columns
    if "_CI_" in col
]

table5_df.loc[
    table5_df["Validation"] != "Nested_LOSO",
    ci_columns
] = np.nan

table5_df.to_csv(
    FOLDERS["tables"]
    / "Table5_Performance_and_95CI.csv",
    index=False
)


# ============================================================
# 14. PAIRED BOOTSTRAP:
# NGBOOST VS OTHER FOUR MODELS
# ============================================================

print("\nRunning paired bootstrap comparisons...")

paired_records = []

ngb_subset = ordered_predictions["NGBoost"]

actual = ngb_subset["y_true"].to_numpy()
ngb_pred = ngb_subset["y_pred"].to_numpy()

ngb_rmse = np.sqrt(
    np.mean((ngb_pred - actual) ** 2)
)

for model_name in MODELS:

    if model_name == "NGBoost":
        continue

    other_pred = (
        ordered_predictions[model_name]["y_pred"]
        .to_numpy()
    )

    other_rmse = np.sqrt(
        np.mean((other_pred - actual) ** 2)
    )

    differences = []

    for idx in bootstrap_indices:

        rmse_ngb = np.sqrt(
            np.mean(
                (ngb_pred[idx] - actual[idx]) ** 2
            )
        )

        rmse_other = np.sqrt(
            np.mean(
                (other_pred[idx] - actual[idx]) ** 2
            )
        )

        differences.append(
            rmse_ngb - rmse_other
        )

    paired_records.append({
        "Comparison": f"NGBoost_minus_{model_name}",
        "NGBoost_RMSE": ngb_rmse,
        "Other_RMSE": other_rmse,
        "Delta_RMSE": ngb_rmse - other_rmse,
        "CI_Lower": np.percentile(
            differences, 2.5
        ),
        "CI_Upper": np.percentile(
            differences, 97.5
        )
    })

paired_df = pd.DataFrame(paired_records)

paired_df.to_csv(
    FOLDERS["tables"]
    / "Paired_Bootstrap_NGBoost_vs_Others.csv",
    index=False
)


# ============================================================
# 15. PER-SOURCE METRICS
# ============================================================

source_records = []

for model_name in MODELS:

    model_data = outer_df[
        outer_df["model"] == model_name
    ]

    for source, subset in model_data.groupby(
        "source_id"
    ):

        metrics = calculate_metrics(
            subset["y_true"],
            subset["y_pred"]
        )

        source_records.append({
            "Model": model_name,
            "Source_ID": source,
            **metrics
        })

source_df = pd.DataFrame(source_records)

source_df.to_csv(
    FOLDERS["supplement"]
    / "Per_Source_Performance.csv",
    index=False
)


# ============================================================
# 16. TABLE 6: RESIDUAL DISTRIBUTION
# ============================================================

table6_records = []

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    residuals = predicted - actual

    table6_records.append({
        "Model": model_name,
        "N": len(residuals),
        "Mean": np.mean(residuals),
        "SD": np.std(residuals, ddof=1),
        "Minimum": np.min(residuals),
        "P25": np.percentile(residuals, 25),
        "Median": np.median(residuals),
        "P75": np.percentile(residuals, 75),
        "Maximum": np.max(residuals),
        "Normalized_SD": (
            np.std(residuals, ddof=1)
            / np.std(actual, ddof=1)
        ),
        "Skewness": skew(
            residuals,
            bias=False
        )
    })

table6_df = pd.DataFrame(table6_records)

table6_df.to_csv(
    FOLDERS["tables"]
    / "Table6_Residual_Distribution.csv",
    index=False
)


# ============================================================
# 17. TRAINING-FIT METRICS
# ============================================================

training_metrics = []

for model_name in MODELS:

    subset = full_df[
        full_df["model"] == model_name
    ]

    metrics = calculate_metrics(
        subset["y_true"],
        subset["y_pred"]
    )

    training_metrics.append({
        "Model": model_name,
        **metrics
    })

training_metrics_df = pd.DataFrame(
    training_metrics
)

training_metrics_df.to_csv(
    FOLDERS["supplement"]
    / "Full_Database_Training_Fit_Metrics.csv",
    index=False
)


# ============================================================
# 18. TWO-PANEL FIGURES: FIVE MODELS
# ============================================================

print("\nGenerating figures...")

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    residuals = predicted - actual

    source_codes = pd.Categorical(
        subset["source_id"]
    ).codes

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5.5)
    )

    lower = min(
        actual.min(),
        predicted.min()
    )

    upper = max(
        actual.max(),
        predicted.max()
    )

    line = np.linspace(
        lower,
        upper,
        300
    )

    # Panel A
    axes[0].scatter(
        actual,
        predicted,
        c=source_codes,
        cmap="tab20",
        s=40,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.25
    )

    axes[0].plot(
        line,
        line,
        "k-",
        linewidth=1.5,
        label="Ideal prediction"
    )

    axes[0].plot(
        line,
        1.2 * line,
        "r--",
        linewidth=1.2,
        label="±20% error"
    )

    axes[0].plot(
        line,
        0.8 * line,
        "r--",
        linewidth=1.2
    )

    axes[0].set_xlabel(
        "Measured compressive strength (MPa)",
        fontsize=12
    )

    axes[0].set_ylabel(
        "Predicted compressive strength (MPa)",
        fontsize=12
    )

    axes[0].set_title(
        f"(a) {model_name}: Predicted vs measured",
        fontsize=13
    )

    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.2)

    # Panel B
    axes[1].scatter(
        actual,
        residuals,
        c=source_codes,
        cmap="tab20",
        s=40,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.25
    )

    axes[1].axhline(
        0,
        color="black",
        linewidth=1.5
    )

    axes[1].set_xlabel(
        "Measured compressive strength (MPa)",
        fontsize=12
    )

    axes[1].set_ylabel(
        "Residual (predicted − measured), MPa",
        fontsize=12
    )

    axes[1].set_title(
        f"(b) {model_name}: Residuals",
        fontsize=13
    )

    axes[1].grid(alpha=0.2)

    plt.tight_layout()

    plt.savefig(
        FOLDERS["figures"]
        / f"{model_name}_Two_Panel.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 19. SOFTWARE VERSIONS
# ============================================================

def package_version(name):

    try:
        return version(name)
    except PackageNotFoundError:
        return "Not installed"


software = {
    "Python": sys.version,
    "OS": platform.platform(),
    "Processor": platform.processor(),
    "Seed": SEED,
    "Samples": N,
    "Sources": N_SOURCES,
    "Outer_CV": "LeaveOneGroupOut",
    "Inner_CV": "GroupKFold(5)",
    "N_ITER": N_ITER,
    "Bootstrap": N_BOOTSTRAP,
    "Versions": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scipy": package_version("scipy"),
        "scikit-learn": package_version("scikit-learn"),
        "ngboost": package_version("ngboost"),
        "matplotlib": package_version("matplotlib"),
        "joblib": package_version("joblib")
    }
}

with open(
    OUTPUT_DIR / "Software_Versions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        software,
        f,
        indent=4
    )


# ============================================================
# 20. CONSOLIDATED EXCEL WORKBOOK
# ============================================================

excel_path = (
    OUTPUT_DIR
    / "WGP_Five_Models_Complete_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    search_space_df.to_excel(
        writer,
        sheet_name="Table3_Search",
        index=False
    )

    final_params_df.to_excel(
        writer,
        sheet_name="Table3_Final",
        index=False
    )

    table5_df.to_excel(
        writer,
        sheet_name="Table5_Performance",
        index=False
    )

    table6_df.to_excel(
        writer,
        sheet_name="Table6_Residuals",
        index=False
    )

    paired_df.to_excel(
        writer,
        sheet_name="Paired_Bootstrap",
        index=False
    )

    source_df.to_excel(
        writer,
        sheet_name="Per_Source",
        index=False
    )

    training_metrics_df.to_excel(
        writer,
        sheet_name="Training_Fit",
        index=False
    )

    outer_df.to_excel(
        writer,
        sheet_name="LOSO_Predictions",
        index=False
    )

    train_df.to_excel(
        writer,
        sheet_name="Train_Predictions",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test_Predictions",
        index=False
    )

    full_df.to_excel(
        writer,
        sheet_name="Full_Fit_Predictions",
        index=False
    )

    tuning_df.to_excel(
        writer,
        sheet_name="Tuning_Log",
        index=False
    )


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("NESTED LOSO RESULTS")
print("=" * 75)

summary = table5_df[
    table5_df["Validation"] == "Nested_LOSO"
]

print(
    summary[
        [
            "Model",
            "R2",
            "RMSE",
            "MAE",
            "MAPE_percent"
        ]
    ].sort_values(
        "RMSE"
    ).to_string(index=False)
)

print("\n" + "=" * 75)
print("ALL FIVE MODELS COMPLETED")
print("=" * 75)

print("Models:", list(MODELS.keys()))
print("Records:", N)
print("Sources:", N_SOURCES)

print("\nResults directory:")
print(OUTPUT_DIR)

print("\nExcel workbook:")
print(excel_path)

print("\nEND OF PROGRAM")


Loading WGP dataset...
Samples: 337
Features: 11
Sources: 18

MODEL: AdaBoost
AdaBoost: LOSO fold 1/18, source=S01
AdaBoost: LOSO fold 2/18, source=S02
AdaBoost: LOSO fold 3/18, source=S03
AdaBoost: LOSO fold 4/18, source=S04
AdaBoost: LOSO fold 5/18, source=S05
AdaBoost: LOSO fold 6/18, source=S06
AdaBoost: LOSO fold 7/18, source=S07
AdaBoost: LOSO fold 8/18, source=S08
AdaBoost: LOSO fold 9/18, source=S09
AdaBoost: LOSO fold 10/18, source=S10
AdaBoost: LOSO fold 11/18, source=S11
AdaBoost: LOSO fold 12/18, source=S12
AdaBoost: LOSO fold 13/18, source=S13
AdaBoost: LOSO fold 14/18, source=S14
AdaBoost: LOSO fold 15/18, source=S15
AdaBoost: LOSO fold 16/18, source=S16
AdaBoost: LOSO fold 17/18, source=S17
AdaBoost: LOSO fold 18/18, source=S18
AdaBoost: Random 80:20 comparison...
AdaBoost: Final full-database tuning...
AdaBoost: COMPLETED in 12.3 minutes

MODEL: Random_Forest
Random_Forest: LOSO fold 1/18, source=S01
Random_Forest: LOSO fold 2/18, source=S02
Random_Forest: LOSO fold 3/

In [3]:
# ============================================================
# WASTE GLASS POWDER CONCRETE
# COMPLETE MODELLING AND MANUSCRIPT REVISION
#
# ADDITIONAL BASELINE: XGBOOST (SINGLE-MODEL RUN)
#
# Reviewer requirements:
# B1-B4, B6-B7
# C1
# D1-D7
#
# Python script: WGP_XGBoost_Same_Structure.py
# ============================================================

import sys
import os
import time
import json
import platform
import warnings
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd
import joblib

from scipy.stats import skew

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import (
    LeaveOneGroupOut,
    GroupKFold,
    RandomizedSearchCV,
    train_test_split,
    ParameterGrid
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_PATH = Path(
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work"
    r"\Data\01_WGP_database_with_ids.csv"
)

OUTPUT_DIR = Path(
    r"D:\2026 Work\My Papers\waste glass concrete"
    r"\Revision of Manuscript\1st Revision\Work\Models_XGBoost"
)

SEED = 42

# Parallelize independent cross-validation fits; limit workers to avoid RAM pressure.
# Adjust to 2 for an 8 GB computer; 4 for 16 GB or more.
CV_WORKERS = min(4, max(1, (os.cpu_count() or 2) - 1))
# Prevent BLAS/NumPy from starting multiple threads in every worker.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# Final manuscript settings
N_ITER = 30
N_BOOTSTRAP = 2000

# For preliminary testing, use:
# N_ITER = 5
# N_BOOTSTRAP = 100

TARGET = "CS"
SOURCE = "source_id"
RECORD_ID = "record_id"

FEATURES = [
    "GGP size",
    "Replacement",
    "Cement",
    "Max size",
    "CA",
    "FA",
    "WC Ratio",
    "SiO2",
    "CaO",
    "Na2O",
    "Days"
]

np.random.seed(SEED)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)


# ============================================================
# 2. OUTPUT FOLDERS
# ============================================================

FOLDERS = {
    "models": OUTPUT_DIR / "Saved_Models",
    "predictions": OUTPUT_DIR / "Predictions",
    "tables": OUTPUT_DIR / "Tables",
    "figures": OUTPUT_DIR / "Figures",
    "supplement": OUTPUT_DIR / "Supplementary",
    "logs": OUTPUT_DIR / "Logs"
}

for folder in FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. LOAD DATA
# ============================================================

print("\nLoading WGP dataset...")

df = pd.read_csv(DATA_PATH)

required = FEATURES + [
    TARGET,
    SOURCE,
    RECORD_ID
]

missing = [
    col for col in required
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

if df[RECORD_ID].isna().any():
    raise ValueError("Missing record IDs detected.")

if df[RECORD_ID].duplicated().any():
    raise ValueError("Duplicate record IDs detected.")

if df[SOURCE].isna().any():
    raise ValueError("Missing source IDs detected.")

for col in FEATURES + [TARGET]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

if df[TARGET].isna().any():
    raise ValueError("Target contains missing values.")

if df[FEATURES].isna().all().any():
    raise ValueError(
        "At least one input feature is entirely missing."
    )

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[SOURCE].copy()

N = len(df)
N_SOURCES = groups.nunique()

print("Samples:", N)
print("Features:", len(FEATURES))
print("Sources:", N_SOURCES)

if N != 337:
    raise ValueError(
        f"Expected 337 records; found {N}."
    )

if N_SOURCES != 18:
    raise ValueError(
        f"Expected 18 sources; found {N_SOURCES}."
    )


# ============================================================
# 4. XGBOOST MODEL (same pipeline and tuning conventions)
# ============================================================

def make_pipeline(model, scale=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

MODELS = {
    "XGBoost": {
        "pipeline": make_pipeline(XGBRegressor(
            objective="reg:squarederror",
            random_state=SEED,
            n_jobs=1,
            tree_method="hist",
            verbosity=0
        )),
        "params": {
            "model__n_estimators": [100, 200, 400, 600],
            "model__max_depth": [2, 3, 4, 5],
            "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
            "model__min_child_weight": [1, 3, 5],
            "model__reg_lambda": [1, 5, 10]
        }
    }
}


# ============================================================
# 5. METRICS
# ============================================================

def calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    error = predicted - actual

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    mae = mean_absolute_error(
        actual, predicted
    )

    if len(actual) > 1 and np.var(actual) > 0:
        r2 = r2_score(actual, predicted)
    else:
        r2 = np.nan

    valid = np.abs(actual) > 1e-12

    if valid.any():

        relative_error = (
            np.abs(error[valid])
            / np.abs(actual[valid])
        )

        mape = 100 * np.mean(relative_error)

        within10 = 100 * np.mean(
            relative_error <= 0.10
        )

        within20 = 100 * np.mean(
            relative_error <= 0.20
        )

        within30 = 100 * np.mean(
            relative_error <= 0.30
        )

    else:

        mape = np.nan
        within10 = np.nan
        within20 = np.nan
        within30 = np.nan

    return {
        "N": len(actual),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE_percent": mape,
        "Within_10_percent": within10,
        "Within_20_percent": within20,
        "Within_30_percent": within30,
        "Max_Overprediction": max(
            0.0, float(np.max(error))
        ),
        "Max_Underprediction": min(
            0.0, float(np.min(error))
        ),
        "Residual_Mean": float(np.mean(error)),
        "Residual_SD": (
            float(np.std(error, ddof=1))
            if len(error) > 1
            else np.nan
        ),
        "Residual_Skewness": (
            float(skew(error, bias=False))
            if len(error) > 2
            and np.std(error) > 0
            else np.nan
        )
    }


# ============================================================
# 6. HYPERPARAMETER TUNING
# ============================================================

def tune_model(
    configuration,
    X_train,
    y_train,
    group_train
):

    pipeline = clone(
        configuration["pipeline"]
    )

    params = configuration["params"]

    inner_cv = GroupKFold(
        n_splits=5
    )

    total_combinations = len(
        ParameterGrid(params)
    )

    candidates = min(
        N_ITER,
        total_combinations
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=candidates,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=SEED,
        n_jobs=CV_WORKERS,
        pre_dispatch=CV_WORKERS,
        refit=True,
        error_score="raise",
        return_train_score=False
    )

    search.fit(
        X_train,
        y_train,
        groups=group_train
    )

    return search


# ============================================================
# 7. PREDICTION RECORD FUNCTION
# ============================================================

def prediction_records(
    model_name,
    indices,
    predictions,
    split_name,
    fold=None,
    mu=None,
    sigma=None
):

    records = []

    for j, idx in enumerate(indices):

        actual = float(y.iloc[idx])
        predicted = float(predictions[j])

        records.append({
            "record_id": df[RECORD_ID].iloc[idx],
            "source_id": groups.iloc[idx],
            "model": model_name,
            "split": split_name,
            "outer_fold": fold,
            "y_true": actual,
            "y_pred": predicted,
            "error_pred_minus_actual": (
                predicted - actual
            ),
            "absolute_error": abs(
                predicted - actual
            ),
            "mu": (
                float(mu[j])
                if mu is not None
                else np.nan
            ),
            "sigma": (
                float(sigma[j])
                if sigma is not None
                else np.nan
            )
        })

    return records


# ============================================================
# 8. MAIN MODELLING
# ============================================================

outer_records = []
train_records = []
test_records = []
full_fit_records = []

tuning_records = []
search_space_records = []
final_parameter_records = []

outer_cv = LeaveOneGroupOut()

for model_name, configuration in MODELS.items():

    model_start = time.perf_counter()
    print("\n" + "=" * 70)
    print("MODEL:", model_name)
    print("=" * 70)

    # --------------------------------------------------------
    # TABLE 3: SEARCH SPACE
    # --------------------------------------------------------

    total_combinations = len(
        ParameterGrid(configuration["params"])
    )

    for param, values in configuration["params"].items():

        search_space_records.append({
            "Model": model_name,
            "Hyperparameter": param,
            "Search_Values": str(values),
            "Total_Grid_Combinations": total_combinations,
            "Candidates_Per_Search": min(
                N_ITER,
                total_combinations
            )
        })

    # --------------------------------------------------------
    # NESTED LOSO
    # --------------------------------------------------------

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y, groups),
        start=1
    ):

        heldout_source = groups.iloc[test_idx[0]]

        print(
            f"{model_name}: LOSO fold "
            f"{fold}/{N_SOURCES}, "
            f"source={heldout_source}",
            flush=True
        )

        search = tune_model(
            configuration,
            X.iloc[train_idx],
            y.iloc[train_idx],
            groups.iloc[train_idx]
        )

        fitted = search.best_estimator_

        predicted = fitted.predict(
            X.iloc[test_idx]
        )

        mu = None
        sigma = None

        if model_name == "NGBoost":

            transformed = fitted[:-1].transform(
                X.iloc[test_idx]
            )

            distribution = (
                fitted.named_steps["model"]
                .pred_dist(transformed)
            )

            mu = np.asarray(
                distribution.loc
            )

            sigma = np.asarray(
                distribution.scale
            )

        outer_records.extend(
            prediction_records(
                model_name,
                test_idx,
                predicted,
                "Nested_LOSO",
                fold=fold,
                mu=mu,
                sigma=sigma
            )
        )

        tuning_records.append({
            "Model": model_name,
            "Validation": "Nested_LOSO",
            "Outer_Fold": fold,
            "Heldout_Source": heldout_source,
            "Training_N": len(train_idx),
            "Testing_N": len(test_idx),
            "Best_Inner_RMSE": -search.best_score_,
            "Candidates_Evaluated": len(
                search.cv_results_["params"]
            ),
            "Best_Parameters": json.dumps(
                search.best_params_,
                default=str
            )
        })

    # --------------------------------------------------------
    # RANDOM 80:20 SPLIT
    # --------------------------------------------------------

    print(
        f"{model_name}: Random 80:20 comparison...",
        flush=True
    )

    indices = np.arange(N)

    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.20,
        random_state=SEED,
        shuffle=True
    )

    search_80 = tune_model(
        configuration,
        X.iloc[train_idx],
        y.iloc[train_idx],
        groups.iloc[train_idx]
    )

    fitted_80 = search_80.best_estimator_

    predicted_train = fitted_80.predict(
        X.iloc[train_idx]
    )

    predicted_test = fitted_80.predict(
        X.iloc[test_idx]
    )

    train_records.extend(
        prediction_records(
            model_name,
            train_idx,
            predicted_train,
            "Random_80_Training"
        )
    )

    test_records.extend(
        prediction_records(
            model_name,
            test_idx,
            predicted_test,
            "Random_20_Testing"
        )
    )

    tuning_records.append({
        "Model": model_name,
        "Validation": "Random_80_20",
        "Outer_Fold": np.nan,
        "Heldout_Source": np.nan,
        "Training_N": len(train_idx),
        "Testing_N": len(test_idx),
        "Best_Inner_RMSE": -search_80.best_score_,
        "Candidates_Evaluated": len(
            search_80.cv_results_["params"]
        ),
        "Best_Parameters": json.dumps(
            search_80.best_params_,
            default=str
        )
    })

    # --------------------------------------------------------
    # FINAL FULL-DATABASE MODEL
    # --------------------------------------------------------

    print(
        f"{model_name}: Final full-database tuning...",
        flush=True
    )

    final_search = tune_model(
        configuration,
        X,
        y,
        groups
    )

    final_model = final_search.best_estimator_

    joblib.dump(
        final_model,
        FOLDERS["models"]
        / f"{model_name}_Final_Pipeline.joblib"
    )

    final_parameter_records.append({
        "Model": model_name,
        "Best_Inner_RMSE": -final_search.best_score_,
        "Candidates_Evaluated": len(
            final_search.cv_results_["params"]
        ),
        "Final_Selected_Parameters": json.dumps(
            final_search.best_params_,
            default=str
        )
    })

    full_predictions = final_model.predict(X)

    full_fit_records.extend(
        prediction_records(
            model_name,
            np.arange(N),
            full_predictions,
            "Full_Database_Training_Fit"
        )
    )

    # Persist this model's predictions and tuning logs immediately.
    pd.DataFrame([r for r in outer_records if r["model"] == model_name]).to_csv(
        FOLDERS["predictions"] / f"{model_name}_LOSO_checkpoint.csv", index=False
    )
    pd.DataFrame([r for r in tuning_records if r["Model"] == model_name]).to_csv(
        FOLDERS["logs"] / f"{model_name}_tuning_checkpoint.csv", index=False
    )
    print(f"{model_name}: COMPLETED in {(time.perf_counter()-model_start)/60:.1f} minutes", flush=True)


# ============================================================
# 9. SAVE ALL PREDICTIONS
# ============================================================

outer_df = pd.DataFrame(outer_records)
train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)
full_df = pd.DataFrame(full_fit_records)

outer_df.to_csv(
    FOLDERS["predictions"]
    / "outer_fold_predictions.csv",
    index=False
)

train_df.to_csv(
    FOLDERS["predictions"]
    / "random80_train_actual_predicted_error.csv",
    index=False
)

test_df.to_csv(
    FOLDERS["predictions"]
    / "random20_test_actual_predicted_error.csv",
    index=False
)

full_df.to_csv(
    FOLDERS["predictions"]
    / "full_database_training_fit_predictions.csv",
    index=False
)

search_space_df = pd.DataFrame(
    search_space_records
)

final_params_df = pd.DataFrame(
    final_parameter_records
)

tuning_df = pd.DataFrame(
    tuning_records
)

search_space_df.to_csv(
    FOLDERS["tables"]
    / "Table3_Search_Spaces.csv",
    index=False
)

final_params_df.to_csv(
    FOLDERS["tables"]
    / "Table3_Final_Selected_Parameters.csv",
    index=False
)

tuning_df.to_csv(
    FOLDERS["logs"]
    / "Hyperparameter_Tuning_Log.csv",
    index=False
)


# ============================================================
# 10. TABLE 5: PERFORMANCE
# ============================================================

table5_records = []

for model_name in MODELS:

    for validation, data in [
        ("Nested_LOSO", outer_df),
        ("Random_80_Training", train_df),
        ("Random_20_Testing", test_df)
    ]:

        subset = data[
            data["model"] == model_name
        ]

        metrics = calculate_metrics(
            subset["y_true"],
            subset["y_pred"]
        )

        table5_records.append({
            "Validation": validation,
            "Model": model_name,
            **metrics
        })

table5_df = pd.DataFrame(table5_records)


# ============================================================
# 11. SOURCE-CLUSTER BOOTSTRAP
# ============================================================

print("\nRunning cluster bootstrap...")

rng = np.random.default_rng(SEED)

unique_sources = groups.unique()

source_indices = {
    source: np.flatnonzero(
        groups.to_numpy() == source
    )
    for source in unique_sources
}

bootstrap_indices = []

for b in range(N_BOOTSTRAP):

    sampled_sources = rng.choice(
        unique_sources,
        size=N_SOURCES,
        replace=True
    )

    sampled_indices = np.concatenate([
        source_indices[source]
        for source in sampled_sources
    ])

    bootstrap_indices.append(sampled_indices)


# ============================================================
# 12. ALIGN LOSO PREDICTIONS
# ============================================================

ordered_predictions = {}

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    subset = (
        subset.set_index("record_id")
        .loc[df[RECORD_ID]]
    )

    ordered_predictions[model_name] = subset


# ============================================================
# 13. BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

ci_records = []

for model_name in MODELS:

    subset = ordered_predictions[model_name]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    bootstrap_metrics = {
        "R2": [],
        "RMSE": [],
        "MAE": []
    }

    for idx in bootstrap_indices:

        metrics = calculate_metrics(
            actual[idx],
            predicted[idx]
        )

        for metric in bootstrap_metrics:
            bootstrap_metrics[metric].append(
                metrics[metric]
            )

    result = {"Model": model_name}

    for metric, values in bootstrap_metrics.items():

        result[f"{metric}_CI_Lower"] = (
            np.nanpercentile(values, 2.5)
        )

        result[f"{metric}_CI_Upper"] = (
            np.nanpercentile(values, 97.5)
        )

    ci_records.append(result)

ci_df = pd.DataFrame(ci_records)

table5_df = table5_df.merge(
    ci_df,
    on="Model",
    how="left"
)

ci_columns = [
    col for col in table5_df.columns
    if "_CI_" in col
]

table5_df.loc[
    table5_df["Validation"] != "Nested_LOSO",
    ci_columns
] = np.nan

table5_df.to_csv(
    FOLDERS["tables"]
    / "Table5_Performance_and_95CI.csv",
    index=False
)


# ============================================================
# 14. PAIRED BOOTSTRAP
# ============================================================
# A paired comparison requires the original models' aligned LOSO predictions.
# Do not manufacture comparison values in a single-model XGBoost run.
paired_df = pd.DataFrame(columns=["Comparison", "Delta_RMSE", "CI_Lower", "CI_Upper"])
paired_df.to_csv(FOLDERS["tables"] / "Paired_Bootstrap_Not_Run.csv", index=False)


# ============================================================
# 15. PER-SOURCE METRICS
# ============================================================

source_records = []

for model_name in MODELS:

    model_data = outer_df[
        outer_df["model"] == model_name
    ]

    for source, subset in model_data.groupby(
        "source_id"
    ):

        metrics = calculate_metrics(
            subset["y_true"],
            subset["y_pred"]
        )

        source_records.append({
            "Model": model_name,
            "Source_ID": source,
            **metrics
        })

source_df = pd.DataFrame(source_records)

source_df.to_csv(
    FOLDERS["supplement"]
    / "Per_Source_Performance.csv",
    index=False
)


# ============================================================
# 16. TABLE 6: RESIDUAL DISTRIBUTION
# ============================================================

table6_records = []

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    residuals = predicted - actual

    table6_records.append({
        "Model": model_name,
        "N": len(residuals),
        "Mean": np.mean(residuals),
        "SD": np.std(residuals, ddof=1),
        "Minimum": np.min(residuals),
        "P25": np.percentile(residuals, 25),
        "Median": np.median(residuals),
        "P75": np.percentile(residuals, 75),
        "Maximum": np.max(residuals),
        "Normalized_SD": (
            np.std(residuals, ddof=1)
            / np.std(actual, ddof=1)
        ),
        "Skewness": skew(
            residuals,
            bias=False
        )
    })

table6_df = pd.DataFrame(table6_records)

table6_df.to_csv(
    FOLDERS["tables"]
    / "Table6_Residual_Distribution.csv",
    index=False
)


# ============================================================
# 17. TRAINING-FIT METRICS
# ============================================================

training_metrics = []

for model_name in MODELS:

    subset = full_df[
        full_df["model"] == model_name
    ]

    metrics = calculate_metrics(
        subset["y_true"],
        subset["y_pred"]
    )

    training_metrics.append({
        "Model": model_name,
        **metrics
    })

training_metrics_df = pd.DataFrame(
    training_metrics
)

training_metrics_df.to_csv(
    FOLDERS["supplement"]
    / "Full_Database_Training_Fit_Metrics.csv",
    index=False
)


# ============================================================
# 18. TWO-PANEL FIGURES: FIVE MODELS
# ============================================================

print("\nGenerating figures...")

for model_name in MODELS:

    subset = outer_df[
        outer_df["model"] == model_name
    ]

    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()

    residuals = predicted - actual

    source_codes = pd.Categorical(
        subset["source_id"]
    ).codes

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13, 5.5)
    )

    lower = min(
        actual.min(),
        predicted.min()
    )

    upper = max(
        actual.max(),
        predicted.max()
    )

    line = np.linspace(
        lower,
        upper,
        300
    )

    # Panel A
    axes[0].scatter(
        actual,
        predicted,
        c=source_codes,
        cmap="tab20",
        s=40,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.25
    )

    axes[0].plot(
        line,
        line,
        "k-",
        linewidth=1.5,
        label="Ideal prediction"
    )

    axes[0].plot(
        line,
        1.2 * line,
        "r--",
        linewidth=1.2,
        label="±20% error"
    )

    axes[0].plot(
        line,
        0.8 * line,
        "r--",
        linewidth=1.2
    )

    axes[0].set_xlabel(
        "Measured compressive strength (MPa)",
        fontsize=12
    )

    axes[0].set_ylabel(
        "Predicted compressive strength (MPa)",
        fontsize=12
    )

    axes[0].set_title(
        f"(a) {model_name}: Predicted vs measured",
        fontsize=13
    )

    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.2)

    # Panel B
    axes[1].scatter(
        actual,
        residuals,
        c=source_codes,
        cmap="tab20",
        s=40,
        alpha=0.8,
        edgecolors="black",
        linewidths=0.25
    )

    axes[1].axhline(
        0,
        color="black",
        linewidth=1.5
    )

    axes[1].set_xlabel(
        "Measured compressive strength (MPa)",
        fontsize=12
    )

    axes[1].set_ylabel(
        "Residual (predicted − measured), MPa",
        fontsize=12
    )

    axes[1].set_title(
        f"(b) {model_name}: Residuals",
        fontsize=13
    )

    axes[1].grid(alpha=0.2)

    plt.tight_layout()

    plt.savefig(
        FOLDERS["figures"]
        / f"{model_name}_Two_Panel.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)


# ============================================================
# 18B. RANDOM 20% TEST-ONLY FIGURE (68 OBSERVATIONS)
# ============================================================
for model_name in MODELS:
    subset = test_df[test_df["model"] == model_name]
    actual = subset["y_true"].to_numpy()
    predicted = subset["y_pred"].to_numpy()
    residuals = predicted - actual
    lo = min(actual.min(), predicted.min())
    hi = max(actual.max(), predicted.max())
    line = np.linspace(lo, hi, 300)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    axes[0].scatter(actual, predicted, s=45, alpha=0.8, edgecolors="black", linewidths=0.3)
    axes[0].plot(line, line, "k-", label="Ideal prediction")
    axes[0].plot(line, 1.2 * line, "r--", label="±20% error")
    axes[0].plot(line, 0.8 * line, "r--")
    axes[0].set(xlabel="Measured compressive strength (MPa)", ylabel="Predicted compressive strength (MPa)", title=f"(a) {model_name}: Random 20% test (n={len(actual)})")
    axes[0].legend()
    axes[0].grid(alpha=0.2)
    axes[1].scatter(actual, residuals, s=45, alpha=0.8, edgecolors="black", linewidths=0.3)
    axes[1].axhline(0, color="black")
    axes[1].set(xlabel="Measured compressive strength (MPa)", ylabel="Residual (predicted − measured), MPa", title=f"(b) {model_name}: Test residuals")
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(FOLDERS["figures"] / f"{model_name}_Random20_Test_Two_Panel.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(residuals, bins="auto", edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", linestyle="--", label="Zero error")
    ax.set(xlabel="Residual (predicted − measured), MPa", ylabel="Frequency", title=f"{model_name}: Random 20% test residual distribution (n={len(actual)})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FOLDERS["figures"] / f"{model_name}_Random20_Test_Error_Distribution.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# 19. SOFTWARE VERSIONS
# ============================================================

def package_version(name):

    try:
        return version(name)
    except PackageNotFoundError:
        return "Not installed"


software = {
    "Python": sys.version,
    "OS": platform.platform(),
    "Processor": platform.processor(),
    "Seed": SEED,
    "Samples": N,
    "Sources": N_SOURCES,
    "Outer_CV": "LeaveOneGroupOut",
    "Inner_CV": "GroupKFold(5)",
    "N_ITER": N_ITER,
    "Bootstrap": N_BOOTSTRAP,
    "Versions": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "scipy": package_version("scipy"),
        "scikit-learn": package_version("scikit-learn"),
        "xgboost": package_version("xgboost"),
        "matplotlib": package_version("matplotlib"),
        "joblib": package_version("joblib")
    }
}

with open(
    OUTPUT_DIR / "Software_Versions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        software,
        f,
        indent=4
    )


# ============================================================
# 20. CONSOLIDATED EXCEL WORKBOOK
# ============================================================

excel_path = (
    OUTPUT_DIR
    / "WGP_XGBoost_Complete_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    search_space_df.to_excel(
        writer,
        sheet_name="Table3_Search",
        index=False
    )

    final_params_df.to_excel(
        writer,
        sheet_name="Table3_Final",
        index=False
    )

    table5_df.to_excel(
        writer,
        sheet_name="Table5_Performance",
        index=False
    )

    table6_df.to_excel(
        writer,
        sheet_name="Table6_Residuals",
        index=False
    )

    paired_df.to_excel(
        writer,
        sheet_name="Paired_Bootstrap",
        index=False
    )

    source_df.to_excel(
        writer,
        sheet_name="Per_Source",
        index=False
    )

    training_metrics_df.to_excel(
        writer,
        sheet_name="Training_Fit",
        index=False
    )

    outer_df.to_excel(
        writer,
        sheet_name="LOSO_Predictions",
        index=False
    )

    train_df.to_excel(
        writer,
        sheet_name="Train_Predictions",
        index=False
    )

    test_df.to_excel(
        writer,
        sheet_name="Test_Predictions",
        index=False
    )

    full_df.to_excel(
        writer,
        sheet_name="Full_Fit_Predictions",
        index=False
    )

    tuning_df.to_excel(
        writer,
        sheet_name="Tuning_Log",
        index=False
    )


# ============================================================
# 21. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("NESTED LOSO RESULTS")
print("=" * 75)

summary = table5_df[
    table5_df["Validation"] == "Nested_LOSO"
]

print(
    summary[
        [
            "Model",
            "R2",
            "RMSE",
            "MAE",
            "MAPE_percent"
        ]
    ].sort_values(
        "RMSE"
    ).to_string(index=False)
)

print("\n" + "=" * 75)
print("XGBOOST MODEL COMPLETED")
print("=" * 75)

print("Models:", list(MODELS.keys()))
print("Records:", N)
print("Sources:", N_SOURCES)

print("\nResults directory:")
print(OUTPUT_DIR)

print("\nExcel workbook:")
print(excel_path)

print("\nEND OF PROGRAM")


Loading WGP dataset...
Samples: 337
Features: 11
Sources: 18

MODEL: XGBoost
XGBoost: LOSO fold 1/18, source=S01
XGBoost: LOSO fold 2/18, source=S02
XGBoost: LOSO fold 3/18, source=S03
XGBoost: LOSO fold 4/18, source=S04
XGBoost: LOSO fold 5/18, source=S05
XGBoost: LOSO fold 6/18, source=S06
XGBoost: LOSO fold 7/18, source=S07
XGBoost: LOSO fold 8/18, source=S08
XGBoost: LOSO fold 9/18, source=S09
XGBoost: LOSO fold 10/18, source=S10
XGBoost: LOSO fold 11/18, source=S11
XGBoost: LOSO fold 12/18, source=S12
XGBoost: LOSO fold 13/18, source=S13
XGBoost: LOSO fold 14/18, source=S14
XGBoost: LOSO fold 15/18, source=S15
XGBoost: LOSO fold 16/18, source=S16
XGBoost: LOSO fold 17/18, source=S17
XGBoost: LOSO fold 18/18, source=S18
XGBoost: Random 80:20 comparison...
XGBoost: Final full-database tuning...
XGBoost: COMPLETED in 5.1 minutes

Running cluster bootstrap...

Generating figures...

NESTED LOSO RESULTS
  Model       R2     RMSE      MAE  MAPE_percent
XGBoost 0.300869 9.864008 7.98400